In [1]:

# 必要なライブラリをインストール
!pip -q install selenium webdriver-manager beautifulsoup4 pandas

# Google Chromeをインストール
!apt-get update -qq
!apt-get install -y -qq wget gnupg

!wget -q -O /tmp/google-chrome.deb \
https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb

!apt-get install -y -qq /tmp/google-chrome.deb


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package at-spi2-common.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../0-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatk1.0-0t64:amd64.
Preparing to unpack .../1-libatk1.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk1.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../2-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libatk-bridge2.0-0t64:amd64.
Preparing to unpack .../3-libatk-bridge2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatk-bridge2.0-0t64:amd64 (2.52.0-1b

In [2]:

import re
import time
from datetime import datetime

from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait

from webdriver_manager.chrome import ChromeDriverManager


# ==========================================
# 1. HTMLから株価データを抽出する関数
# ==========================================

def extract_stock_data(html):

    soup = BeautifulSoup(html, "html.parser")

    text = soup.get_text(" ", strip=True)

    pattern = (
        r"(\d{4}年\d{1,2}月\d{1,2}日)\s*"
        r"C\s*([\d,]+)\s*"
        r"L\s*([\d,]+)\s*"
        r"H\s*([\d,]+)\s*"
        r"O\s*([\d,]+)"
    )

    match = re.search(pattern, text)

    if match is None:
        return None

    date = datetime.strptime(
        match.group(1), "%Y年%m月%d日"
    ).strftime("%Y-%m-%d")

    close = int(match.group(2).replace(",", ""))
    low = int(match.group(3).replace(",", ""))
    high = int(match.group(4).replace(",", ""))
    open_price = int(match.group(5).replace(",", ""))

    return [date, open_price, high, low, close]


# ==========================================
# 2. 株価データを取得する関数
# ==========================================

def get_stock_values(driver, url):

    driver.get(url)

    wait = WebDriverWait(driver, 30)

    # グラフの表示を待機
    chart = wait.until(
        lambda d: next(
            (
                e for e in d.find_elements(
                    By.CSS_SELECTOR,
                    'canvas.modcharts-panel-root[aria-label*="NIKKEI"]'
                )
                if e.is_displayed() and e.size["width"] > 0
            ),
            None
        )
    )

    width = int(chart.size["width"])

    print("グラフの幅：", width)

    # 株価情報の表示を待機
    wait.until(
        lambda d: extract_stock_data(
            d.page_source
        ) is not None
    )

    # グラフ中央へマウスを移動
    ActionChains(driver).move_to_element(
        chart
    ).perform()

    # グラフ右端付近へ移動
    right_offset = width // 2 - 20

    ActionChains(driver).move_by_offset(
        right_offset, 0
    ).perform()

    time.sleep(0.5)

    stock_values = []
    recorded_dates = set()

    # 1pxずつ左へ移動
    for i in range(width - 40):

        ActionChains(driver).move_by_offset(
            -1, 0
        ).perform()

        html = driver.page_source

        data = extract_stock_data(html)

        if data is not None:

            date = data[0]

            if date not in recorded_dates:

                stock_values.append(data)
                recorded_dates.add(date)

    # 日付の古い順に並べ替え
    stock_values.sort(key=lambda x: x[0])

    return stock_values


# ==========================================
# 3. main処理
# ==========================================

def main():

    url = (
        "https://www.nikkei.com/smartchart/"
        "?code=N101/T&timeframe=6m"
        "&interval=1Day&eventsShow=1"
    )

    # Chromeの設定
    options = Options()

    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")

    driver = None

    # 処理時間の計測開始
    start_time = datetime.now()
    start = time.perf_counter()

    try:

        # Chromeを起動
        service = Service(
            ChromeDriverManager().install()
        )

        driver = webdriver.Chrome(
            service=service,
            options=options
        )

        print("スクレイピング開始：", start_time)

        # 株価データを取得
        stock_values = get_stock_values(
            driver, url
        )

        # 結果を表示
        print("\n===== スクレイピング結果 =====")
        print("日付, 始値, 高値, 安値, 終値")

        for data in stock_values:

            print(
                data[0],
                data[1],
                data[2],
                data[3],
                data[4],
                sep=", "
            )

        print("\n取得件数：", len(stock_values))

    finally:

        # Chromeを終了
        if driver is not None:
            driver.quit()

        # 処理時間の計測終了
        end_time = datetime.now()
        end = time.perf_counter()

        elapsed_time = end - start

        print("\n===== 処理時間 =====")
        print("開始時間：", start_time)
        print("終了時間：", end_time)
        print(f"処理時間：{elapsed_time:.2f}秒")


# ==========================================
# 4. プログラム実行
# ==========================================

if __name__ == "__main__":
    main()


スクレイピング開始： 2026-11-22 13:17:00.237091
グラフの幅： 1533

===== スクレイピング結果 =====
日付, 始値, 高値, 安値, 終値
2026-05-26, 65247, 65318, 64606, 64996
2026-05-27, 65778, 66429, 64999, 64999
2026-05-28, 64771, 65166, 63875, 64693
2026-05-29, 65134, 66505, 65134, 66330
2026-06-01, 66363, 67231, 66245, 66934
2026-06-02, 66630, 66748, 65551, 66734
2026-06-03, 67239, 68786, 67239, 68402
2026-06-04, 67861, 68052, 66921, 67471
2026-06-05, 67115, 67115, 65862, 66588
2026-06-08, 65948, 66115, 63407, 64025
2026-06-09, 64625, 65485, 63919, 65417
2026-06-10, 64952, 65099, 63733, 64179
2026-06-11, 63329, 64396, 62336, 64217
2026-06-12, 65176, 67066, 64998, 66020
2026-06-15, 66783, 69682, 66783, 69318
2026-06-16, 69289, 70021, 69096, 69405
2026-06-17, 69006, 70126, 68986, 69902
2026-06-18, 70164, 71399, 70093, 71053
2026-06-19, 71551, 71953, 70518, 71250
2026-06-22, 71067, 72832, 71010, 72354
2026-06-23, 72404, 72618, 69788, 69788
2026-06-24, 69615, 70219, 68461, 69175
2026-06-25, 70114, 72594, 69983, 72366
2026-06-26,